# Megane で分子を描いて、機械学習ポテンシャルで構造最適化する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megane-labs/megane/blob/main/examples/colab_ml_optimization.ipynb)

この notebook では次のことをします。

1. **Megane Builder** (`megane.Builder`) で分子を描く
2. 機械学習ポテンシャル **MACE** (ASE calculator) で構造最適化する
3. **最適化前を半透明**にして最適化後と重ね、構造変化を Megane で見る
4. (おまけ) 最適化の軌跡をアニメーションで再生する

Google Colab でもローカルの Jupyter / JupyterLab / VS Code でも動きます。GPU ランタイムがあれば自動で使いますが、小さな分子なら CPU でも数秒〜数十秒で終わります。

## 0. インストール

`megane` (ビューア + Builder)、`ase` (構造最適化)、`mace-torch` (機械学習ポテンシャル) を入れます。Colab には PyTorch が最初から入っています。

In [ ]:
%pip install -q megane ase mace-torch

In [ ]:
# In Colab, third-party widgets (anywidget) need the custom widget manager.
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass  # not on Colab

import warnings

import numpy as np
import torch
from ase.build import minimize_rotation_and_translation, molecule
from ase.io import write
from ase.optimize import BFGS

import megane

warnings.filterwarnings("ignore")  # torch / e3nn deprecation noise from MACE

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"megane {megane.__version__}, device = {device}")

## 1. Megane Builder で分子を描く

下のセルを実行すると Builder が開きます。分子の作り方はいくつかあります。

- **SMILES から作る**: 左のツールバーで **Place** (六角形のアイコン、キー `P`) を選ぶ → 右の Details パネルで **Choose a molecule…** → 一覧の下の **Sketch…** → **Paste MOL** を押して、SMILES (例: アスピリン `CC(=O)Oc1ccccc1C(=O)O`) か MOL ファイルの中身を貼る → **Add to library** → 一覧に増えた分子の **Add** を押す (または **Place** を押してからキャンバスをクリック)。RDKit (WebAssembly) が 3D 構造を作り、水素も付けてくれます。notebook 版の Builder には 2D スケッチャー (Ketcher) は入っていないので、SMILES / MOL の貼り付けを使います。
- **ライブラリのプリセット** (水、メタノール、エタノール、ベンゼン、…) を同じ一覧から **Add** する
- **Add atom** / **Bond** ツールで原子を 1 個ずつ描く

**ポイント**: ライブラリに追加した分子は RDKit が MMFF94s で軽く最適化済みなので、そのままだと ML ポテンシャルで最適化しても変化は小さめです。変化をはっきり見たいときは、**Move** ツールで原子をドラッグして構造を崩しておくか、次の節の `RATTLE` で乱数で崩してください。

何も描かずに先に進むと、代わりにエタノールを使います。

In [ ]:
builder = megane.Builder(height="560px")
builder

## 2. Builder の構造を ASE の `Atoms` として取り出す

Builder は編集のたびに構造を Python 側に書き戻すので、`builder.to_ase()` で画面上の構造をそのまま取り出せます。

In [ ]:
# Random displacement (Å standard deviation) applied before optimizing, so the
# change is easy to see. Set to 0.0 to optimize the drawn structure as is.
RATTLE = 0.15

if builder.n_atoms > 0:
    atoms_init = builder.to_ase()
    print(f"Builder から {builder.n_atoms} 原子 / {builder.n_bonds} 結合を読み込みました")
else:
    atoms_init = molecule("CH3CH2OH")
    print("Builder が空なので、代わりにエタノールを使います")

# A molecule in vacuum: no periodic boundary conditions.
if not atoms_init.cell.any():
    atoms_init.pbc = False

if RATTLE > 0:
    atoms_init.rattle(stdev=RATTLE, seed=0)

print(atoms_init.get_chemical_formula(), f"({len(atoms_init)} atoms)")

## 3. 機械学習ポテンシャルを用意する

[MACE](https://github.com/ACEsuit/mace) の学習済みモデルを ASE calculator として使います。初回はモデル (数十 MB) を自動でダウンロードします。

| `MODEL` | 中身 | 向いている系 | ライセンス |
| --- | --- | --- | --- |
| `"mace-off"` | MACE-OFF23 (small) | 有機分子 (H, C, N, O, F, P, S, Cl, Br, I) | ASL (学術・非商用) |
| `"mace-mp"` | MACE-MP-0 (small) | 周期表のほぼ全元素・無機材料 | MIT |

有機分子なら `"mace-off"`、金属や無機を含むなら `"mace-mp"` を選んでください。

In [ ]:
from mace.calculators import mace_mp, mace_off

MODEL = "mace-off"  # or "mace-mp"

if MODEL == "mace-off":
    calc = mace_off(model="small", device=device, default_dtype="float64")
else:
    calc = mace_mp(model="small", device=device, default_dtype="float64")

## 4. 構造最適化

BFGS で、全原子にかかる力が `FMAX` (eV/Å) 以下になるまで最適化します。各ステップの構造は `opt.traj` に保存されます (最後の節で再生します)。

In [ ]:
FMAX = 0.01  # eV/Å
MAX_STEPS = 500

# Energy of the starting structure.
atoms_init.calc = calc
e_init = atoms_init.get_potential_energy()

atoms_opt = atoms_init.copy()
atoms_opt.calc = calc
opt = BFGS(atoms_opt, trajectory="opt.traj", logfile="-")
converged = opt.run(fmax=FMAX, steps=MAX_STEPS)
e_opt = atoms_opt.get_potential_energy()

print()
print(f"収束: {converged} ({opt.get_number_of_steps()} steps)")
print(f"E(最適化前) = {e_init:.4f} eV")
print(f"E(最適化後) = {e_opt:.4f} eV")
print(f"ΔE          = {e_opt - e_init:.4f} eV")

## 5. 最適化前を半透明にして重ねて表示する

並進・回転で最適化後の構造を最適化前に重ね合わせて (Kabsch)、RMSD と大きく動いた原子を確認したあと、2 つの構造を 1 つのビューに重ねます。

Megane のパイプラインでは、`LoadStructure` を 2 つ同じ `Viewport` につなぐと 2 つのレイヤーとして重ねて描けます。最適化前の原子と結合をそれぞれ `Modify(opacity=...)` に通して半透明にしています。

```
before.xyz ─ LoadStructure ─┬─────────── Modify(opacity) ─┐ particle
                            └─ AddBonds ─ Modify(opacity) ─┤ bond
after.xyz  ─ LoadStructure ─┬──────────────────────────────┤ particle   Viewport
                            └─ AddBonds ───────────────────┘ bond
```

In [ ]:
# Superimpose the optimized structure onto the initial one (rotation + translation only).
minimize_rotation_and_translation(atoms_init, atoms_opt)

disp = np.linalg.norm(atoms_opt.positions - atoms_init.positions, axis=1)
rmsd = np.sqrt(np.mean(disp**2))
print(f"RMSD = {rmsd:.3f} Å, 最大変位 = {disp.max():.3f} Å")
print("大きく動いた原子:")
for i in np.argsort(disp)[::-1][:5]:
    print(f"  #{i:<3d} {atoms_init[i].symbol:<2s} {disp[i]:.3f} Å")

write("before.xyz", atoms_init)
write("after.xyz", atoms_opt)

In [ ]:
def compare_viewer(before: str, after: str, *, before_opacity: float = 0.3) -> megane.MolecularViewer:
    """Overlay two structures; *before* is drawn semi-transparent."""
    pipe = megane.Pipeline()
    view = pipe.add_node(megane.Viewport())

    # Before optimization: atoms and bonds both through Modify(opacity).
    s0 = pipe.add_node(megane.LoadStructure(before))
    b0 = pipe.add_node(megane.AddBonds(source="distance"))
    atoms_ghost = pipe.add_node(megane.Modify(opacity=before_opacity))
    bonds_ghost = pipe.add_node(megane.Modify(opacity=before_opacity))
    pipe.add_edge(s0.out.particle, atoms_ghost.inp.particle)
    pipe.add_edge(atoms_ghost.out.particle, view.inp.particle)
    pipe.add_edge(s0.out.particle, b0.inp.particle)
    # Modify accepts bond data on the same port and returns bonds.
    pipe.add_edge(b0.out.bond, bonds_ghost.inp.particle)
    pipe.add_edge(bonds_ghost.out.particle, view.inp.bond)

    # After optimization: drawn as is.
    s1 = pipe.add_node(megane.LoadStructure(after))
    b1 = pipe.add_node(megane.AddBonds(source="distance"))
    pipe.add_edge(s1.out.particle, view.inp.particle)
    pipe.add_edge(s1.out.particle, b1.inp.particle)
    pipe.add_edge(b1.out.bond, view.inp.bond)

    viewer = megane.MolecularViewer()
    viewer.set_pipeline(pipe)
    return viewer


compare_viewer("before.xyz", "after.xyz", before_opacity=0.3)

半透明のほうが **最適化前**、不透明のほうが **最適化後** です。ドラッグで回転、スクロールでズームできます。`before_opacity` を変えると透け具合を調整できます (0 で非表示、1 で不透明)。

最適化後の構造は `after.xyz` に保存されています。Builder に戻して続きを編集することもできます: `builder.load("after.xyz")`。

## 6. (おまけ) 最適化の軌跡を再生する

`opt.traj` (ASE trajectory) をそのまま Megane で開くと、最適化の各ステップをスライダーで再生できます。

In [ ]:
traj_viewer = megane.view_traj("opt.traj")
print(f"{traj_viewer.total_frames} frames")
traj_viewer